In [5]:
import polars as pl
import pandas as pd
import numpy as np
import time
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. CHARGEMENT DES DONNÉES (Fixé avec pheno.txt)
# =====================================================================
print("Étape 1: Chargement des données...")

# Remplacez ceci par le chargement réel de votre matrice PLINK filtrée (p < 0.01)
raw_matrix = pl.read_csv("matrix_plink_data.raw", separator="\t")
df = raw_matrix.to_pandas()

# Chargement des vrais phénotypes
pheno_df = pd.read_csv("pheno.txt", sep="\t")
df['IID'] = df['IID'].astype(str)
pheno_df['IID'] = pheno_df['IID'].astype(str)

# Fusion propre
df = df.drop(columns=['PHENOTYPE'], errors='ignore').merge(pheno_df[['IID', 'PHENOTYPE']], on='IID', how='inner')

# Séparation Features (X) et Target (y)
snp_cols = [col for col in df.columns if col.startswith('exm-') or col.startswith('rs')]
X = df[snp_cols]
# 2 = Fumeur -> 1, 1 = Non-fumeur -> 0
y = df['PHENOTYPE'].apply(lambda val: 1 if val == 2 else 0) 

# =====================================================================
# 2. SÉPARATION TRAIN/TEST (Zéro Fuite de Données)
# =====================================================================
print("Étape 2: Séparation Train/Test...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scaling obligatoire pour LASSO et SVM
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# =====================================================================
# 3. FILTRAGE LASSO (Sélection de Caractéristiques 1)
# =====================================================================
print("\nÉtape 3: Filtrage par régression LASSO...")
lasso = LogisticRegression(penalty='l1', C=0.1, solver='liblinear', class_weight='balanced', random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

# Extraction des gènes survivants
lasso_coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_scaled.columns)
lasso_survivors = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f" -> SNPs ayant survécu au LASSO: {len(lasso_survivors)}")

X_train_lasso = X_train_scaled[lasso_survivors]
X_test_lasso = X_test_scaled[lasso_survivors]

# =====================================================================
# 4. CLASSEMENT RFE (Sélection de Caractéristiques 2)
# =====================================================================
print("\nÉtape 4: Classement RFE (Recursive Feature Elimination)...")
# Nous utilisons un SVM linéaire rapide pour classer l'importance des gènes restants
rfe_estimator = SVC(kernel="linear", class_weight="balanced", random_state=42)

# On élimine 5% des gènes les plus faibles à chaque itération
rfe = RFE(estimator=rfe_estimator, n_features_to_select=1, step=0.05)
rfe.fit(X_train_lasso, y_train)

# Création d'une liste triée des meilleurs gènes
ranked_features_df = pd.DataFrame({
    'Feature': X_train_lasso.columns,
    'Rank': rfe.ranking_
}).sort_values(by='Rank')

ranked_snps = ranked_features_df['Feature'].tolist()

# =====================================================================
# 5. OPTIMISATION ET ÉVALUATION DYNAMIQUE (Top N)
# =====================================================================
print("\nÉtape 5: Évaluation Dynamique (Optuna par Sous-ensemble)...")
optuna.logging.set_verbosity(optuna.logging.WARNING) # Pour ne pas spammer la console

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tune_and_evaluate_svm(top_n):
    # 1. Sélectionner uniquement le Top N des features
    features_n = ranked_snps[:top_n]
    X_train_n = X_train_lasso[features_n]
    X_test_n = X_test_lasso[features_n]
    
    # 2. Fonction d'objectif Optuna spécifique à cette dimension
    def objective(trial):
        c_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
        gamma_val = trial.suggest_float("gamma", 1e-4, 1.0, log=True)
        
        model = SVC(kernel="rbf", C=c_val, gamma=gamma_val, class_weight="balanced", random_state=42)
        scores = cross_val_score(model, X_train_n, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
        return scores.mean()
    
    # 3. Lancer l'étude Optuna (50 itérations suffisent avec le sampler TPE)
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=50, n_jobs=-1) # Exploite tous vos cœurs !
    
    best_params = study.best_params
    
    # 4. Entraîner le modèle final avec les meilleurs paramètres
    final_model = SVC(
        kernel="rbf", 
        C=best_params["C"], 
        gamma=best_params["gamma"], 
        class_weight="balanced", 
        probability=True, 
        random_state=42
    )
    final_model.fit(X_train_n, y_train)
    
    # 5. Évaluer sur le vrai Test Set
    preds = final_model.predict_proba(X_test_n)[:, 1]
    auc = roc_auc_score(y_test, preds)
    
    return auc, best_params

# Évaluation des différents seuils comme dans le papier
thresholds = [50, 100, 200, 300, 400, 500]

print("=====================================================================")
print(" RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (AUC HONNÊTE)")
print("=====================================================================")

for n in thresholds:
    # On s'assure de ne pas demander plus de features que ce qui a survécu au LASSO
    actual_n = min(n, len(ranked_snps))
    
    start_time = time.time()
    auc, params = tune_and_evaluate_svm(actual_n)
    elapsed = time.time() - start_time
    
    print(f"Top {actual_n} SNPs Model Performance:")
    print(f" -> Test AUC: {auc:.3f}")
    print(f" -> Meilleurs Hyperparamètres: C={params['C']:.4f}, gamma={params['gamma']:.4f}")
    print(f" -> Temps d'optimisation: {elapsed:.1f} secondes\n")

Étape 1: Chargement des données...
Étape 2: Séparation Train/Test...

Étape 3: Filtrage par régression LASSO...
 -> SNPs ayant survécu au LASSO: 1447

Étape 4: Classement RFE (Recursive Feature Elimination)...

Étape 5: Évaluation Dynamique (Optuna par Sous-ensemble)...
 RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (AUC HONNÊTE)
Top 50 SNPs Model Performance:
 -> Test AUC: 0.509
 -> Meilleurs Hyperparamètres: C=0.7963, gamma=0.0041
 -> Temps d'optimisation: 4.5 secondes

Top 100 SNPs Model Performance:
 -> Test AUC: 0.525
 -> Meilleurs Hyperparamètres: C=7.2288, gamma=0.0018
 -> Temps d'optimisation: 4.5 secondes

Top 200 SNPs Model Performance:
 -> Test AUC: 0.529
 -> Meilleurs Hyperparamètres: C=2.7755, gamma=0.0020
 -> Temps d'optimisation: 9.7 secondes

Top 300 SNPs Model Performance:
 -> Test AUC: 0.502
 -> Meilleurs Hyperparamètres: C=7.5550, gamma=0.0004
 -> Temps d'optimisation: 14.2 secondes

Top 400 SNPs Model Performance:
 -> Test AUC: 0.511
 -> Meilleurs Hyperparamètres: C=5.803

In [6]:
import polars as pl
import pandas as pd
import numpy as np
import time
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE, SelectFpr, chi2
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. CHARGEMENT DES DONNÉES
# =====================================================================
print("Étape 1: Chargement des données...")
# Chargement de la matrice brute contenant TOUS les SNPs
raw_matrix = pl.read_csv("matrix_plink_data.raw", separator="\t")
df = raw_matrix.to_pandas()

pheno_df = pd.read_csv("pheno.txt", sep="\t")
df['IID'] = df['IID'].astype(str)
pheno_df['IID'] = pheno_df['IID'].astype(str)

df = df.drop(columns=['PHENOTYPE'], errors='ignore').merge(pheno_df[['IID', 'PHENOTYPE']], on='IID', how='inner')

snp_cols = [col for col in df.columns if col.startswith('exm-') or col.startswith('rs')]
X = df[snp_cols]
y = df['PHENOTYPE'].apply(lambda val: 1 if val == 2 else 0) 

# =====================================================================
# 2. SÉPARATION TRAIN/TEST (Le bouclier anti-fuite)
# =====================================================================
print("Étape 2: Séparation Train/Test...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =====================================================================
# 3. FILTRE UNIVARIÉ CHI-CARRÉ (La clé manquante !)
# =====================================================================
print("\nÉtape 3: Filtrage univarié Chi-Carré (p < 0.01)...")
# Ceci élimine l'océan de bruit avant que les modèles complexes n'interviennent
chi2_selector = SelectFpr(score_func=chi2, alpha=0.01)

# FIT UNIQUEMENT SUR LE TRAIN
X_train_chi2_array = chi2_selector.fit_transform(X_train, y_train)
X_test_chi2_array = chi2_selector.transform(X_test)

# Récupérer les noms des SNPs qui ont survécu
surviving_snps = X_train.columns[chi2_selector.get_support()]
print(f" -> SNPs ayant survécu au Chi-Carré: {len(surviving_snps)}")

X_train_chi2 = pd.DataFrame(X_train_chi2_array, columns=surviving_snps, index=X_train.index)
X_test_chi2 = pd.DataFrame(X_test_chi2_array, columns=surviving_snps, index=X_test.index)

# SCALING OBLIGATOIRE APRÈS LE CHI-CARRÉ (Pour LASSO et SVM)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_chi2), columns=surviving_snps, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_chi2), columns=surviving_snps, index=X_test.index)

# =====================================================================
# 4. FILTRAGE LASSO
# =====================================================================
print("\nÉtape 4: Filtrage multivarié par régression LASSO...")
lasso = LogisticRegression(penalty='l1', C=0.1, solver='liblinear', class_weight='balanced', random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

lasso_coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_scaled.columns)
lasso_survivors = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f" -> SNPs ayant survécu au LASSO: {len(lasso_survivors)}")

X_train_lasso = X_train_scaled[lasso_survivors]
X_test_lasso = X_test_scaled[lasso_survivors]

# =====================================================================
# 5. CLASSEMENT RFE
# =====================================================================
print("\nÉtape 5: Classement RFE (Recursive Feature Elimination)...")
if len(lasso_survivors) > 0:
    rfe_estimator = SVC(kernel="linear", class_weight="balanced", random_state=42)
    rfe = RFE(estimator=rfe_estimator, n_features_to_select=1, step=0.05)
    rfe.fit(X_train_lasso, y_train)

    ranked_features_df = pd.DataFrame({
        'Feature': X_train_lasso.columns,
        'Rank': rfe.ranking_
    }).sort_values(by='Rank')

    ranked_snps = ranked_features_df['Feature'].tolist()
else:
    print("ERREUR: Le LASSO a tout supprimé. Augmentez la valeur de C dans le LASSO (ex: C=0.5 ou C=1.0).")
    ranked_snps = []

# =====================================================================
# 6. OPTIMISATION ET ÉVALUATION DYNAMIQUE
# =====================================================================
print("\nÉtape 6: Évaluation Dynamique (Optuna par Sous-ensemble)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tune_and_evaluate_svm(top_n):
    features_n = ranked_snps[:top_n]
    X_train_n = X_train_lasso[features_n]
    X_test_n = X_test_lasso[features_n]
    
    def objective(trial):
        c_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
        gamma_val = trial.suggest_float("gamma", 1e-4, 1.0, log=True)
        
        model = SVC(kernel="rbf", C=c_val, gamma=gamma_val, class_weight="balanced", random_state=42)
        scores = cross_val_score(model, X_train_n, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
        return scores.mean()
    
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=50, n_jobs=-1) 
    
    best_params = study.best_params
    
    final_model = SVC(
        kernel="rbf", 
        C=best_params["C"], 
        gamma=best_params["gamma"], 
        class_weight="balanced", 
        probability=True, 
        random_state=42
    )
    final_model.fit(X_train_n, y_train)
    
    preds = final_model.predict_proba(X_test_n)[:, 1]
    auc = roc_auc_score(y_test, preds)
    
    return auc, best_params

thresholds = [50, 100, 200, 300, 400, 500]

print("=====================================================================")
print(" RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (AUC HONNÊTE)")
print("=====================================================================")

if len(ranked_snps) > 0:
    for n in thresholds:
        actual_n = min(n, len(ranked_snps))
        
        start_time = time.time()
        auc, params = tune_and_evaluate_svm(actual_n)
        elapsed = time.time() - start_time
        
        print(f"Top {actual_n} SNPs Model Performance:")
        print(f" -> Test AUC: {auc:.3f}")
        print(f" -> Meilleurs Hyperparamètres: C={params['C']:.4f}, gamma={params['gamma']:.4f}")
        print(f" -> Temps d'optimisation: {elapsed:.1f} secondes\n")

Étape 1: Chargement des données...
Étape 2: Séparation Train/Test...

Étape 3: Filtrage univarié Chi-Carré (p < 0.01)...
 -> SNPs ayant survécu au Chi-Carré: 28

Étape 4: Filtrage multivarié par régression LASSO...
 -> SNPs ayant survécu au LASSO: 21

Étape 5: Classement RFE (Recursive Feature Elimination)...

Étape 6: Évaluation Dynamique (Optuna par Sous-ensemble)...
 RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (AUC HONNÊTE)
Top 21 SNPs Model Performance:
 -> Test AUC: 0.527
 -> Meilleurs Hyperparamètres: C=7.0076, gamma=0.0003
 -> Temps d'optimisation: 2.4 secondes

Top 21 SNPs Model Performance:
 -> Test AUC: 0.529
 -> Meilleurs Hyperparamètres: C=0.8199, gamma=0.0030
 -> Temps d'optimisation: 2.5 secondes

Top 21 SNPs Model Performance:
 -> Test AUC: 0.529
 -> Meilleurs Hyperparamètres: C=0.8627, gamma=0.0029
 -> Temps d'optimisation: 2.5 secondes

Top 21 SNPs Model Performance:
 -> Test AUC: 0.532
 -> Meilleurs Hyperparamètres: C=1.4627, gamma=0.0028
 -> Temps d'optimisation: 2.5 sec

In [8]:
from sklearn.feature_selection import SelectKBest, f_classif

# =====================================================================
# 3. FILTRE UNIVARIÉ ANOVA (Le bon test mathématique)
# =====================================================================
print("\nÉtape 3: Filtrage univarié ANOVA F-test...")
# On sélectionne les 3000 meilleurs SNPs pour garantir que le modèle a assez de matière
anova_selector = SelectKBest(score_func=f_classif, k=3000)

# FIT UNIQUEMENT SUR LE TRAIN (Zéro fuite)
X_train_anova_array = anova_selector.fit_transform(X_train, y_train)
X_test_anova_array = anova_selector.transform(X_test)

# Récupérer les noms des SNPs
surviving_snps = X_train.columns[anova_selector.get_support()]
print(f" -> SNPs ayant survécu à l'ANOVA: {len(surviving_snps)}")

X_train_anova = pd.DataFrame(X_train_anova_array, columns=surviving_snps, index=X_train.index)
X_test_anova = pd.DataFrame(X_test_anova_array, columns=surviving_snps, index=X_test.index)

# SCALING
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_anova), columns=surviving_snps, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_anova), columns=surviving_snps, index=X_test.index)

# =====================================================================
# 4. FILTRAGE LASSO
# =====================================================================
print("\nÉtape 4: Filtrage multivarié par régression LASSO...")
# On augmente légèrement le 'C' (1.0 au lieu de 0.1) pour être moins brutal, 
# puisqu'on part de 3000 SNPs propres au lieu de 45000.
lasso = LogisticRegression(penalty='l1', C=1.0, solver='liblinear', class_weight='balanced', random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

lasso_coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_scaled.columns)
lasso_survivors = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f" -> SNPs ayant survécu au LASSO: {len(lasso_survivors)}")

X_train_lasso = X_train_scaled[lasso_survivors]
X_test_lasso = X_test_scaled[lasso_survivors]



# =====================================================================
# 5. CLASSEMENT RFE
# =====================================================================
print("\nÉtape 5: Classement RFE (Recursive Feature Elimination)...")
if len(lasso_survivors) > 0:
    rfe_estimator = SVC(kernel="linear", class_weight="balanced", random_state=42)
    rfe = RFE(estimator=rfe_estimator, n_features_to_select=1, step=0.05)
    rfe.fit(X_train_lasso, y_train)

    ranked_features_df = pd.DataFrame({
        'Feature': X_train_lasso.columns,
        'Rank': rfe.ranking_
    }).sort_values(by='Rank')

    ranked_snps = ranked_features_df['Feature'].tolist()
else:
    print("ERREUR: Le LASSO a tout supprimé. Augmentez la valeur de C dans le LASSO (ex: C=0.5 ou C=1.0).")
    ranked_snps = []

# =====================================================================
# 6. OPTIMISATION ET ÉVALUATION DYNAMIQUE
# =====================================================================
print("\nÉtape 6: Évaluation Dynamique (Optuna par Sous-ensemble)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tune_and_evaluate_svm(top_n):
    features_n = ranked_snps[:top_n]
    X_train_n = X_train_lasso[features_n]
    X_test_n = X_test_lasso[features_n]
    
    def objective(trial):
        c_val = trial.suggest_float("C", 1e-3, 10.0, log=True)
        gamma_val = trial.suggest_float("gamma", 1e-4, 1.0, log=True)
        
        model = SVC(kernel="rbf", C=c_val, gamma=gamma_val, class_weight="balanced", random_state=42)
        scores = cross_val_score(model, X_train_n, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
        return scores.mean()
    
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    study.optimize(objective, n_trials=50, n_jobs=-1) 
    
    best_params = study.best_params
    
    final_model = SVC(
        kernel="rbf", 
        C=best_params["C"], 
        gamma=best_params["gamma"], 
        class_weight="balanced", 
        probability=True, 
        random_state=42
    )
    final_model.fit(X_train_n, y_train)
    
    preds = final_model.predict_proba(X_test_n)[:, 1]
    auc = roc_auc_score(y_test, preds)
    
    return auc, best_params

thresholds = [50, 100, 200, 300, 400, 500]

print("=====================================================================")
print(" RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (AUC HONNÊTE)")
print("=====================================================================")

if len(ranked_snps) > 0:
    for n in thresholds:
        actual_n = min(n, len(ranked_snps))
        
        start_time = time.time()
        auc, params = tune_and_evaluate_svm(actual_n)
        elapsed = time.time() - start_time
        
        print(f"Top {actual_n} SNPs Model Performance:")
        print(f" -> Test AUC: {auc:.3f}")
        print(f" -> Meilleurs Hyperparamètres: C={params['C']:.4f}, gamma={params['gamma']:.4f}")
        print(f" -> Temps d'optimisation: {elapsed:.1f} secondes\n")


Étape 3: Filtrage univarié ANOVA F-test...
 -> SNPs ayant survécu à l'ANOVA: 3000

Étape 4: Filtrage multivarié par régression LASSO...
 -> SNPs ayant survécu au LASSO: 1487

Étape 5: Classement RFE (Recursive Feature Elimination)...

Étape 6: Évaluation Dynamique (Optuna par Sous-ensemble)...
 RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST (AUC HONNÊTE)
Top 50 SNPs Model Performance:
 -> Test AUC: 0.529
 -> Meilleurs Hyperparamètres: C=2.1829, gamma=0.0014
 -> Temps d'optimisation: 5.4 secondes

Top 100 SNPs Model Performance:
 -> Test AUC: 0.525
 -> Meilleurs Hyperparamètres: C=6.3694, gamma=0.0015
 -> Temps d'optimisation: 4.6 secondes

Top 200 SNPs Model Performance:
 -> Test AUC: 0.548
 -> Meilleurs Hyperparamètres: C=9.1560, gamma=0.0009
 -> Temps d'optimisation: 9.7 secondes

Top 300 SNPs Model Performance:
 -> Test AUC: 0.567
 -> Meilleurs Hyperparamètres: C=3.2457, gamma=0.0027
 -> Temps d'optimisation: 16.4 secondes

Top 400 SNPs Model Performance:
 -> Test AUC: 0.571
 -> Meilleurs

In [10]:
from sklearn.feature_selection import SelectKBest, f_classif

# =====================================================================
# 3. FILTRE UNIVARIÉ ANOVA (Le bon test mathématique)
# =====================================================================
print("\nÉtape 3: Filtrage univarié ANOVA F-test...")
# On sélectionne les 3000 meilleurs SNPs pour garantir que le modèle a assez de matière
anova_selector = SelectKBest(score_func=f_classif, k=3000)

# FIT UNIQUEMENT SUR LE TRAIN (Zéro fuite)
X_train_anova_array = anova_selector.fit_transform(X_train, y_train)
X_test_anova_array = anova_selector.transform(X_test)

# Récupérer les noms des SNPs
surviving_snps = X_train.columns[anova_selector.get_support()]
print(f" -> SNPs ayant survécu à l'ANOVA: {len(surviving_snps)}")

X_train_anova = pd.DataFrame(X_train_anova_array, columns=surviving_snps, index=X_train.index)
X_test_anova = pd.DataFrame(X_test_anova_array, columns=surviving_snps, index=X_test.index)

# SCALING
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_anova), columns=surviving_snps, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_anova), columns=surviving_snps, index=X_test.index)

# =====================================================================
# 4. FILTRAGE LASSO
# =====================================================================
print("\nÉtape 4: Filtrage multivarié par régression LASSO...")
# On augmente légèrement le 'C' (1.0 au lieu de 0.1) pour être moins brutal, 
# puisqu'on part de 3000 SNPs propres au lieu de 45000.
lasso = LogisticRegression(penalty='l1', C=0.05, solver='liblinear', class_weight='balanced', random_state=42, max_iter=5000)
lasso.fit(X_train_scaled, y_train)

lasso_coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_scaled.columns)
lasso_survivors = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f" -> SNPs ayant survécu au LASSO: {len(lasso_survivors)}")

X_train_lasso = X_train_scaled[lasso_survivors]
X_test_lasso = X_test_scaled[lasso_survivors]



# =====================================================================
# 5. CLASSEMENT RFE
# =====================================================================
print("\nÉtape 5: Classement RFE (Recursive Feature Elimination)...")
if len(lasso_survivors) > 0:
    rfe_estimator = SVC(kernel="linear", class_weight="balanced", random_state=42)
    rfe = RFE(estimator=rfe_estimator, n_features_to_select=1, step=0.05)
    rfe.fit(X_train_lasso, y_train)

    ranked_features_df = pd.DataFrame({
        'Feature': X_train_lasso.columns,
        'Rank': rfe.ranking_
    }).sort_values(by='Rank')

    ranked_snps = ranked_features_df['Feature'].tolist()
else:
    print("ERREUR: Le LASSO a tout supprimé. Augmentez la valeur de C dans le LASSO (ex: C=0.5 ou C=1.0).")
    ranked_snps = []

# =====================================================================
# 6. OPTIMISATION ET ÉVALUATION DYNAMIQUE
# =====================================================================
from sklearn.ensemble import RandomForestClassifier

print("\nÉtape 6: Évaluation Dynamique (Random Forest + Optuna)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tune_and_evaluate_rf(top_n):
    features_n = ranked_snps[:top_n]
    X_train_n = X_train_lasso[features_n]
    X_test_n = X_test_lasso[features_n]
    
    def objective(trial):
        # Paramètres spécifiques au Random Forest
        n_estimators = trial.suggest_int('n_estimators', 100, 800)
        max_depth = trial.suggest_int('max_depth', 3, 25)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        
        model = RandomForestClassifier(
            n_estimators=n_estimators, 
            max_depth=max_depth, 
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced", 
            random_state=42, 
            n_jobs=2 # Garde de la place pour les threads Optuna
        )
        scores = cross_val_score(model, X_train_n, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
        return scores.mean()
    
    study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
    # 30 itérations suffisent pour RF
    study.optimize(objective, n_trials=30, n_jobs=-1) 
    
    best_params = study.best_params
    
    final_model = RandomForestClassifier(
        n_estimators=best_params["n_estimators"], 
        max_depth=best_params["max_depth"], 
        min_samples_leaf=best_params["min_samples_leaf"],
        class_weight="balanced", 
        random_state=42,
        n_jobs=-1
    )
    final_model.fit(X_train_n, y_train)
    
    preds = final_model.predict_proba(X_test_n)[:, 1]
    auc = roc_auc_score(y_test, preds)
    
    return auc, best_params

thresholds = [50, 100, 200, 300, 400, 500]

print("=====================================================================")
print(" RÉSULTATS FINAUX RANDOM FOREST SUR L'ENSEMBLE DE TEST")
print("=====================================================================")

if len(ranked_snps) > 0:
    for n in thresholds:
        actual_n = min(n, len(ranked_snps))
        
        start_time = time.time()
        auc, params = tune_and_evaluate_rf(actual_n)
        elapsed = time.time() - start_time
        
        print(f"Top {actual_n} SNPs Model Performance:")
        print(f" -> Test AUC: {auc:.3f}")
        print(f" -> Hyperparamètres: Estimators={params['n_estimators']}, Depth={params['max_depth']}, Leaf={params['min_samples_leaf']}")
        print(f" -> Temps d'optimisation: {elapsed:.1f} secondes\n")


Étape 3: Filtrage univarié ANOVA F-test...
 -> SNPs ayant survécu à l'ANOVA: 3000

Étape 4: Filtrage multivarié par régression LASSO...
 -> SNPs ayant survécu au LASSO: 963

Étape 5: Classement RFE (Recursive Feature Elimination)...

Étape 6: Évaluation Dynamique (Random Forest + Optuna)...
 RÉSULTATS FINAUX RANDOM FOREST SUR L'ENSEMBLE DE TEST
Top 50 SNPs Model Performance:
 -> Test AUC: 0.539
 -> Hyperparamètres: Estimators=769, Depth=23, Leaf=4
 -> Temps d'optimisation: 8.5 secondes

Top 100 SNPs Model Performance:
 -> Test AUC: 0.538
 -> Hyperparamètres: Estimators=211, Depth=12, Leaf=7
 -> Temps d'optimisation: 6.6 secondes

Top 200 SNPs Model Performance:
 -> Test AUC: 0.527
 -> Hyperparamètres: Estimators=778, Depth=24, Leaf=10
 -> Temps d'optimisation: 11.4 secondes

Top 300 SNPs Model Performance:
 -> Test AUC: 0.537
 -> Hyperparamètres: Estimators=766, Depth=25, Leaf=9
 -> Temps d'optimisation: 10.8 secondes

Top 400 SNPs Model Performance:
 -> Test AUC: 0.541
 -> Hyperparam

In [12]:
import polars as pl
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. CHARGEMENT ET SÉPARATION
# =====================================================================
print("Étape 1: Chargement des données...")
raw_matrix = pl.read_csv("matrix_plink_data.raw", separator="\t")
df = raw_matrix.to_pandas()

pheno_df = pd.read_csv("pheno.txt", sep="\t")
df['IID'] = df['IID'].astype(str)
pheno_df['IID'] = pheno_df['IID'].astype(str)
df = df.drop(columns=['PHENOTYPE'], errors='ignore').merge(pheno_df[['IID', 'PHENOTYPE']], on='IID', how='inner')

snp_cols = [col for col in df.columns if col.startswith('exm-') or col.startswith('rs')]
X = df[snp_cols]
y = df['PHENOTYPE'].apply(lambda val: 1 if val == 2 else 0) 

print("Étape 2: Séparation Train/Test (Sans fuite)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# =====================================================================
# 3. FILTRE ANOVA LARGEMENT OUVERT (k=5000)
# =====================================================================
print("\nÉtape 3: Filtrage univarié ANOVA F-test (Top 5000)...")
# On garde beaucoup plus de gènes pour construire un score polygénique
anova_selector = SelectKBest(score_func=f_classif, k=5000)

X_train_anova_array = anova_selector.fit_transform(X_train, y_train)
X_test_anova_array = anova_selector.transform(X_test)
surviving_snps = X_train.columns[anova_selector.get_support()]

X_train_anova = pd.DataFrame(X_train_anova_array, columns=surviving_snps, index=X_train.index)
X_test_anova = pd.DataFrame(X_test_anova_array, columns=surviving_snps, index=X_test.index)

# SCALING STRICTEMENT OBLIGATOIRE POUR LA PÉNALITÉ L2
print("Étape 4: Standardisation...")
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_anova), columns=surviving_snps, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_anova), columns=surviving_snps, index=X_test.index)

# =====================================================================
# 5. SCORE DE RISQUE POLYGÉNIQUE (RIDGE REGRESSION / L2) - VERSION RAPIDE
# =====================================================================
print("\nÉtape 5: Entraînement du modèle PRS (Ridge Regression / L2)...")
start_time = time.time()

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# LBFGS est extrêmement rapide pour la pénalité L2 (Ridge).
# On réduit Cs à 10 pour cibler la bonne zone de régularisation sans perdre de temps.
prs_model = LogisticRegressionCV(
    Cs=10, 
    cv=cv_strategy, 
    penalty='l2', 
    solver='lbfgs', # CHANGEMENT CLÉ
    class_weight='balanced', 
    scoring='roc_auc', 
    max_iter=500,   # LBFGS converge généralement en moins de 150 itérations
    random_state=42, 
    n_jobs=-1
)

prs_model.fit(X_train_scaled, y_train)
# =====================================================================
# 6. ÉVALUATION FINALE
# =====================================================================
print("\n=====================================================================")
print(" RÉSULTATS FINAUX : APPROCHE POLYGÉNIQUE HONNÊTE (PRS)")
print("=====================================================================")

# Prédictions sur le Train (pour mesurer le surajustement)
train_preds = prs_model.predict_proba(X_train_scaled)[:, 1]
train_auc = roc_auc_score(y_train, train_preds)

# Prédictions sur le Test (La vérité terrain)
test_preds = prs_model.predict_proba(X_test_scaled)[:, 1]
test_auc = roc_auc_score(y_test, test_preds)

elapsed = time.time() - start_time

print(f" -> Train AUC (Mémorisation) : {train_auc:.3f}")
print(f" -> Test AUC (Vérité Terrain): {test_auc:.3f}")
print(f" -> Meilleur hyperparamètre C trouvé : {prs_model.C_[0]:.4f}")
print(f" -> Temps d'exécution : {elapsed:.1f} secondes\n")

Étape 1: Chargement des données...
Étape 2: Séparation Train/Test (Sans fuite)...

Étape 3: Filtrage univarié ANOVA F-test (Top 5000)...
Étape 4: Standardisation...

Étape 5: Entraînement du modèle PRS (Ridge Regression / L2)...

 RÉSULTATS FINAUX : APPROCHE POLYGÉNIQUE HONNÊTE (PRS)
 -> Train AUC (Mémorisation) : 0.992
 -> Test AUC (Vérité Terrain): 0.548
 -> Meilleur hyperparamètre C trouvé : 0.0008
 -> Temps d'exécution : 5.0 secondes



In [14]:
import polars as pl
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. LOAD DATA & INJECT DEMOGRAPHICS (The Fix)
# =====================================================================
print("Étape 1: Chargement des données et des covariables (Age/Gender)...")

# Load the genetic matrix
raw_matrix = pl.read_csv("matrix_plink_data.raw", separator="\t").to_pandas()
raw_matrix['IID'] = raw_matrix['IID'].astype(str)

# Load the annotation data to get Age, Gender, and Status
annot_data = pl.read_parquet("annot_data_processed.parquet").to_pandas()
annot_data['IID'] = annot_data['Sample'].astype(str)

# Format Target and Demographics
annot_data['PHENOTYPE'] = annot_data['Status'].apply(lambda x: 1 if x == "Smoker" else 0)
annot_data['Gender_Num'] = annot_data['Gender'].apply(lambda x: 1 if x == "Male" else 0)

# Merge Genetics with Demographics
# Merge Genetics with Demographics (Dropping PLINK's placeholder first)
df = raw_matrix.drop(columns=['PHENOTYPE']).merge(
    annot_data[['IID', 'PHENOTYPE', 'Age', 'Gender_Num']], 
    on='IID', 
    how='inner'
)
# Separate column categories
snp_cols = [col for col in df.columns if col.startswith('exm-') or col.startswith('rs')]
covar_cols = ['Age', 'Gender_Num']

X = df[snp_cols + covar_cols]
y = df['PHENOTYPE']

# =====================================================================
# 2. TRAIN/TEST SPLIT (No Leakage)
# =====================================================================
print("Étape 2: Séparation Train/Test...")
# Using random_state=4 to match your original script exactly
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=4)

# Isolate SNPs from Covariates so we don't accidentally filter out Age/Gender
X_train_snps = X_train[snp_cols]
X_test_snps = X_test[snp_cols]

X_train_covars = X_train[covar_cols]
X_test_covars = X_test[covar_cols]

# =====================================================================
# 3. UNIVARIATE FILTERING ON SNPS ONLY
# =====================================================================
print("Étape 3: Filtrage univarié ANOVA (Top 3000 SNPs)...")
anova_selector = SelectKBest(score_func=f_classif, k=3000)

X_train_snps_selected = pd.DataFrame(
    anova_selector.fit_transform(X_train_snps, y_train), 
    columns=X_train_snps.columns[anova_selector.get_support()], 
    index=X_train_snps.index
)
X_test_snps_selected = pd.DataFrame(
    anova_selector.transform(X_test_snps), 
    columns=X_train_snps.columns[anova_selector.get_support()], 
    index=X_test_snps.index
)

# RECOMBINE: Top 3000 SNPs + Age + Gender
X_train_combined = pd.concat([X_train_snps_selected, X_train_covars], axis=1)
X_test_combined = pd.concat([X_test_snps_selected, X_test_covars], axis=1)

# =====================================================================
# 4. SCALING & LASSO FILTERING
# =====================================================================
print("Étape 4: Standardisation et LASSO...")
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_combined), columns=X_train_combined.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_combined), columns=X_test_combined.columns)

# We use LASSO to force the model to weigh the genetics *against* Age and Gender.
lasso = LogisticRegression(penalty='l1', C=0.5, solver='liblinear', class_weight='balanced', random_state=4, max_iter=2000)
lasso.fit(X_train_scaled, y_train)

lasso_coefs = pd.Series(np.abs(lasso.coef_[0]), index=X_train_scaled.columns)
surviving_features = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f" -> Features ayant survécu au LASSO: {len(surviving_features)}")

X_train_final = X_train_scaled[surviving_features]
X_test_final = X_test_scaled[surviving_features]

# =====================================================================
# 5. XGBOOST OPTIMIZATION & EVALUATION
# =====================================================================
print("\nÉtape 5: Évaluation Dynamique (XGBoost + Optuna)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int('n_estimators', 100, 1000),
        "max_depth": trial.suggest_int('max_depth', 2, 20),
        "learning_rate": trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        "subsample": trial.suggest_float('subsample', 0.5, 1.0),
        "min_child_weight": trial.suggest_int('min_child_weight', 1, 10)
    }
    
    # Calculate scale_pos_weight to balance classes automatically
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
    
    model = XGBClassifier(
        **params, 
        scale_pos_weight=ratio,
        objective="binary:logistic", 
        eval_metric="auc", 
        random_state=4, 
        n_jobs=4
    )
    
    scores = cross_val_score(model, X_train_final, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

# Run Optuna Study
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=4))
study.optimize(objective, n_trials=50, n_jobs=-1)

best_params = study.best_params

# Train Final Model
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
final_model = XGBClassifier(
    **best_params, 
    scale_pos_weight=ratio,
    objective="binary:logistic", 
    eval_metric="auc", 
    random_state=4, 
    n_jobs=-1
)
final_model.fit(X_train_final, y_train)

# =====================================================================
# 6. RESULTS
# =====================================================================
print("\n=====================================================================")
print(" RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST")
print("=====================================================================")

train_preds = final_model.predict_proba(X_train_final)[:, 1]
test_preds = final_model.predict_proba(X_test_final)[:, 1]

train_auc = roc_auc_score(y_train, train_preds)
test_auc = roc_auc_score(y_test, test_preds)

print(f" -> Train AUC : {train_auc:.3f}")
print(f" -> Test AUC  : {test_auc:.3f}")
print(f" -> Meilleurs Hyperparamètres : {best_params}")

# Sanity Check: Did Age/Gender survive LASSO?
print("\n--- Importance des Covariables ---")
if 'Age' in surviving_features:
    print(" -> L'Âge a survécu au LASSO et est inclus dans le modèle.")
if 'Gender_Num' in surviving_features:
    print(" -> Le Genre a survécu au LASSO et est inclus dans le modèle.")

Étape 1: Chargement des données et des covariables (Age/Gender)...
Étape 2: Séparation Train/Test...
Étape 3: Filtrage univarié ANOVA (Top 3000 SNPs)...
Étape 4: Standardisation et LASSO...
 -> Features ayant survécu au LASSO: 1423

Étape 5: Évaluation Dynamique (XGBoost + Optuna)...

 RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST
 -> Train AUC : 1.000
 -> Test AUC  : 0.537
 -> Meilleurs Hyperparamètres : {'n_estimators': 876, 'max_depth': 11, 'learning_rate': 0.07740551390757931, 'subsample': 0.6011756632565991, 'min_child_weight': 10}

--- Importance des Covariables ---
 -> Le Genre a survécu au LASSO et est inclus dans le modèle.


In [15]:
import polars as pl
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectFpr, chi2
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. LOAD DATA (STRICTLY GENETICS)
# =====================================================================
print("Étape 1: Chargement des données...")

raw_matrix = pl.read_csv("matrix_plink_data.raw", separator="\t").to_pandas()
raw_matrix['IID'] = raw_matrix['IID'].astype(str)

annot_data = pl.read_parquet("annot_data_processed.parquet").to_pandas()
annot_data['IID'] = annot_data['Sample'].astype(str)

annot_data['PHENOTYPE'] = annot_data['Status'].apply(lambda x: 1 if x == "Smoker" else 0)

# Drop PLINK's placeholder and merge
df = raw_matrix.drop(columns=['PHENOTYPE'], errors='ignore').merge(
    annot_data[['IID', 'PHENOTYPE']], 
    on='IID', 
    how='inner'
)

# Isolate ONLY the SNPs
snp_cols = [col for col in df.columns if col.startswith('exm-') or col.startswith('rs')]
X = df[snp_cols]
y = df['PHENOTYPE']

# =====================================================================
# 2. TRAIN/TEST SPLIT (No Leakage)
# =====================================================================
print("Étape 2: Séparation Train/Test...")
# Stratified to ensure same ratio of smokers/non-smokers in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=4)

# =====================================================================
# 3. CHI-SQUARED FILTERING (Your exact method)
# =====================================================================
print("Étape 3: Filtrage univarié Chi-Carré (alpha=0.01)...")
# We use your exact SelectFpr approach.
chi2_selector = SelectFpr(score_func=chi2, alpha=0.01)

# Fit ONLY on train to prevent data leakage
X_train_chi2 = chi2_selector.fit_transform(X_train, y_train)
X_test_chi2 = chi2_selector.transform(X_test)

surviving_snps = X_train.columns[chi2_selector.get_support()]
print(f" -> SNPs purs ayant survécu au Chi-Carré: {len(surviving_snps)}")

X_train_final = pd.DataFrame(X_train_chi2, columns=surviving_snps)
X_test_final = pd.DataFrame(X_test_chi2, columns=surviving_snps)

# WE SKIP LASSO ENTIRELY HERE to prevent destroying XGBoost's ability to find epistasis.

# =====================================================================
# 4. XGBOOST OPTIMIZATION (Optuna)
# =====================================================================
print("\nÉtape 4: Évaluation Dynamique (XGBoost + Optuna)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int('n_estimators', 200, 1200),
        "max_depth": trial.suggest_int('max_depth', 2, 20),
        "learning_rate": trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
        "subsample": trial.suggest_float('subsample', 0.5, 1.0),
        "min_child_weight": trial.suggest_int('min_child_weight', 1, 10)
    }
    
    # Balance classes dynamically
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
    
    model = XGBClassifier(
        **params, 
        scale_pos_weight=ratio,
        objective="binary:logistic", 
        eval_metric="auc", 
        random_state=4, 
        n_jobs=4
    )
    
    # Ensure cross-validation evaluates the pipeline cleanly
    scores = cross_val_score(model, X_train_final, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=4))
study.optimize(objective, n_trials=50, n_jobs=-1)

best_params = study.best_params

# =====================================================================
# 5. FINAL MODEL EVALUATION
# =====================================================================
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)
final_model = XGBClassifier(
    **best_params, 
    scale_pos_weight=ratio,
    objective="binary:logistic", 
    eval_metric="auc", 
    random_state=4, 
    n_jobs=-1
)
final_model.fit(X_train_final, y_train)

print("\n=====================================================================")
print(" RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST")
print("=====================================================================")

train_preds = final_model.predict_proba(X_train_final)[:, 1]
test_preds = final_model.predict_proba(X_test_final)[:, 1]

train_auc = roc_auc_score(y_train, train_preds)
test_auc = roc_auc_score(y_test, test_preds)

print(f" -> Train AUC : {train_auc:.3f}")
print(f" -> Test AUC  : {test_auc:.3f}")
print(f" -> Meilleurs Hyperparamètres : {best_params}")

Étape 1: Chargement des données...
Étape 2: Séparation Train/Test...
Étape 3: Filtrage univarié Chi-Carré (alpha=0.01)...
 -> SNPs purs ayant survécu au Chi-Carré: 44

Étape 4: Évaluation Dynamique (XGBoost + Optuna)...

 RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST
 -> Train AUC : 0.903
 -> Test AUC  : 0.508
 -> Meilleurs Hyperparamètres : {'n_estimators': 648, 'max_depth': 8, 'learning_rate': 0.0037649239291407963, 'subsample': 0.6233465094967403, 'min_child_weight': 2}


In [19]:
import polars as pl
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectFpr, chi2
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# 1. LOAD THE FULL UNPRUNED DATA (Your exact original logic)
# =====================================================================
print("Étape 1: Chargement des données (cost_matrix_processed.parquet)...")

# Reading and transposing the cost matrix
cost_matrix = pl.read_parquet("cost_matrix_processed.parquet")
gene_col_name = cost_matrix.columns[0] 

cost_matrix = cost_matrix.with_columns(pl.col(gene_col_name).cast(pl.String))
cost_matrix = cost_matrix.transpose(include_header=True, header_name="Sample", column_names=gene_col_name)
cost_matrix = cost_matrix.with_columns(pl.col("Sample").cast(pl.Int32))

# Reading annotation data
annot_data = pl.read_parquet("annot_data_processed.parquet")
annot_data = annot_data.with_columns(pl.col("Sample").cast(pl.Int32))

# Joining data on sample
mdf = cost_matrix.join(annot_data, on="Sample")

# Filtering out ex-smokers and extracting target
mdf = mdf.filter(pl.col("Status") != "Ex-smoker")
y_series = mdf.get_column("Status")
y = (y_series == "Smoker").to_numpy().astype(int)

# Creating binary gender and dropping unused columns
mdf = mdf.with_columns(Gender=(pl.col("Gender") == "Male").cast(pl.Int8))
X_df = mdf.drop(["Sample", "Status"]).to_pandas()

# =====================================================================
# 2. TRAIN/TEST SPLIT
# =====================================================================
print("Étape 2: Séparation Train/Test...")
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=4, stratify=y)

print("Étape 3: MinMax Scaling de l'Age...")
scaler = MinMaxScaler()
X_train['Age'] = scaler.fit_transform(X_train[['Age']])
X_test['Age'] = scaler.transform(X_test[['Age']])

# =====================================================================
# 3. MANUAL MAF FILTERING (Your exact original logic)
# =====================================================================
print("Étape 4: Filtrage Minor Allele Frequency (MAF > 0.01)...")
# Select everything except Age and Gender
snp_cols = [col for col in X_train.columns if col not in ['Age', 'Gender']]
X_train_snps = X_train[snp_cols].values

# Calculate MAF on the training set
N = X_train_snps.shape[0]
AF = X_train_snps.sum(axis=0) / (2 * N)
MAF = np.minimum(AF, 1 - AF)
kept_snp_mask = MAF > 0.01

# Keep the significant SNPs + Age + Gender
kept_snp_cols = np.array(snp_cols)[kept_snp_mask].tolist()
cols_to_keep = kept_snp_cols + ['Age', 'Gender']

X_train_maf = X_train[cols_to_keep]
X_test_maf = X_test[cols_to_keep]
print(f" -> Features restantes après MAF: {X_train_maf.shape[1]}")

# =====================================================================
# 4. CHI-SQUARED FILTERING (Your exact original logic)
# =====================================================================
print("Étape 5: Filtrage Chi-Carré (alpha = 0.01)...")
chi2_selector = SelectFpr(score_func=chi2, alpha=0.01)

# Fit and transform
X_train_chi2_array = chi2_selector.fit_transform(X_train_maf, y_train)
X_test_chi2_array = chi2_selector.transform(X_test_maf)

surviving_chi2_cols = X_train_maf.columns[chi2_selector.get_support()]
print(f" -> Features ayant survécu au Chi-Carré: {len(surviving_chi2_cols)}")

X_train_chi2 = pd.DataFrame(X_train_chi2_array, columns=surviving_chi2_cols)
X_test_chi2 = pd.DataFrame(X_test_chi2_array, columns=surviving_chi2_cols)

# =====================================================================
# 5. LASSO REGRESSION (Your exact original logic)
# =====================================================================
print("Étape 6: Filtrage LASSO...")
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=4)
lambdas_to_try = np.logspace(-4, 2, 100)

logistic_lasso = LogisticRegressionCV(
    cv=cv_strategy,
    Cs=1/lambdas_to_try,
    max_iter=10000,
    random_state=4,
    penalty="l1",
    solver="liblinear",
    scoring='roc_auc',
    n_jobs=-1
)
logistic_lasso.fit(X_train_chi2, y_train)

# Filter by non-zero coefficients
significant_features_mask = logistic_lasso.coef_[0] != 0
surviving_lasso_cols = surviving_chi2_cols[significant_features_mask]

X_train_lasso = X_train_chi2.loc[:, significant_features_mask]
X_test_lasso = X_test_chi2.loc[:, significant_features_mask]
print(f" -> Features ayant survécu au LASSO: {len(surviving_lasso_cols)}")

# =====================================================================
# 6. XGBOOST OPTIMIZATION
# =====================================================================
print("\nÉtape 7: Évaluation Dynamique (XGBoost + Optuna)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "max_depth": trial.suggest_int('max_depth', 1, 20),
        "learning_rate": trial.suggest_float('learning_rate', 0.0001, 1.0, log=True),
        "n_estimators": trial.suggest_int('n_estimators', 200, 2000),
        "subsample": trial.suggest_float('subsample', 0.01, 1.0),
        "min_child_weight": trial.suggest_int('min_child_weight', 1, 15)
    }
    
    model = XGBClassifier(
        **params, 
        objective="binary:logistic", 
        eval_metric="auc", 
        random_state=4, 
        n_jobs=4
    )
    
    # Passing X_train_lasso instead of X_train_chi2 to fix the Optuna bug from your original code
    scores = cross_val_score(model, X_train_lasso, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=4))
study.optimize(objective, n_trials=50, n_jobs=-1)

best_params = study.best_params

final_model = XGBClassifier(
    **best_params, 
    objective="binary:logistic", 
    eval_metric="auc", 
    random_state=4, 
    n_jobs=-1
)
final_model.fit(X_train_lasso, y_train)

print("\n=====================================================================")
print(" RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST")
print("=====================================================================")

train_preds = final_model.predict_proba(X_train_lasso)[:, 1]
test_preds = final_model.predict_proba(X_test_lasso)[:, 1]

print(f" -> Train AUC : {roc_auc_score(y_train, train_preds):.3f}")
print(f" -> Test AUC  : {roc_auc_score(y_test, test_preds):.3f}")
print(f" -> Meilleurs Hyperparamètres : {best_params}")

Étape 1: Chargement des données (cost_matrix_processed.parquet)...
Étape 2: Séparation Train/Test...
Étape 3: MinMax Scaling de l'Age...
Étape 4: Filtrage Minor Allele Frequency (MAF > 0.01)...
 -> Features restantes après MAF: 46323
Étape 5: Filtrage Chi-Carré (alpha = 0.01)...
 -> Features ayant survécu au Chi-Carré: 189
Étape 6: Filtrage LASSO...
 -> Features ayant survécu au LASSO: 189

Étape 7: Évaluation Dynamique (XGBoost + Optuna)...

 RÉSULTATS FINAUX SUR L'ENSEMBLE DE TEST
 -> Train AUC : 0.856
 -> Test AUC  : 0.581
 -> Meilleurs Hyperparamètres : {'max_depth': 1, 'learning_rate': 0.11880570402858154, 'n_estimators': 1535, 'subsample': 0.6727146479948191, 'min_child_weight': 6}


In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

print("--- 1. EXTRACTION DES PHÉNOTYPES ET COVARIABLES ---")
sample_ids, statuses, ages, genders = [], [], [], []

with open("GSE148812_series_matrix.txt", 'r') as f:
    for line in f:
        # Extraire les IDs et le statut fumeur
        if line.startswith("!Sample_title"):
            titles = [x.strip('"\n\r') for x in line.split('\t')[1:]]
            for title in titles:
                if ":" in title:
                    array_id, pheno_string = title.split(":")
                    status = 1 if "non-smoker" not in pheno_string.lower() and "smoker" in pheno_string.lower() else 0
                    sample_ids.append(array_id.strip())
                    statuses.append(status)
                    
        # Extraire l'Âge
        elif line.startswith("!Sample_characteristics_ch1") and "age:" in line.lower():
            ages_raw = [x.strip('"\n\r') for x in line.split('\t')[1:]]
            ages = [float(a.split(':')[1].strip()) if "age:" in a.lower() and a.split(':')[1].strip() != "" else np.nan for a in ages_raw]
            
        # Extraire le Genre (Male = 1, Female = 0 selon ton entraînement)
        elif line.startswith("!Sample_characteristics_ch1") and "gender:" in line.lower():
            genders_raw = [x.strip('"\n\r') for x in line.split('\t')[1:]]
            genders = [1 if "male" in g.lower() and "female" not in g.lower() else 0 for g in genders_raw]

pheno_geo = pd.DataFrame({
    'Sample_ID': sample_ids, 
    'PHENOTYPE_NEW': statuses,
    'Age': ages,
    'Gender': genders
})
pheno_geo.set_index('Sample_ID', inplace=True)
print(f"Chargement réussi : {len(pheno_geo)} labels phénotypiques avec Âge et Genre.")

print("\n--- 2. CHARGEMENT DU DICTIONNAIRE DE PROBES (GPL) ---")
gpl_skip = 0
with open("GPL18545-31886.txt", 'r') as f:
    for i, line in enumerate(f):
        if line.startswith("!dataTable_tableBegin"):
            gpl_skip = i + 1
            break
        elif not line.startswith(('!', '^', '#')) and len(line.split('\t')) > 5:
            gpl_skip = i
            break

gpl_df = pd.read_csv("GPL18545-31886.txt", sep="\t", skiprows=gpl_skip, low_memory=False, on_bad_lines='skip')
probe_to_short = dict(zip(gpl_df['ID'], gpl_df['Name']))
print(f"Dictionnaire chargé : {len(probe_to_short)} sondes.")

print("\n--- 3. CHARGEMENT ET TRANSPOSITION DE LA MATRICE GEO ---")
geo_df = pd.read_csv("GSE148812_genotyping_results.txt", sep="\t", index_col=0)
geo_df = geo_df.T 
# AUCUN RENOMMAGE NÉCESSAIRE. GEO utilise déjà les identifiants longs que XGBoost attend.

print("\n--- 4. ENCODAGE DYNAMIQUE (MAJOR/MINOR) & ALIGNEMENT ---")
from collections import Counter

# Créer une matrice vide correspondant EXACTEMENT aux features que XGBoost attend
geo_numeric = pd.DataFrame(0, index=geo_df.index, columns=surviving_lasso_cols)

matched_count = 0
expected_snps = [col for col in surviving_lasso_cols if col not in ['Age', 'Gender']]

for feature in expected_snps:
    # Recherche du nom EXACT de la feature (sans rien couper)
    if feature in geo_df.columns:
        matched_count += 1
        raw_genotypes = geo_df[feature].astype(str)
        
        # Extraire toutes les lettres valides (A, C, T, G) pour trouver l'allèle majeur/mineur
        # Ignore les appels manquants comme 'NC' ou '--'
        all_letters = "".join([g for g in raw_genotypes if g in ["AA", "AC", "AG", "AT", "CC", "CG", "CT", "GG", "GT", "TT"]])
        
        if len(all_letters) == 0:
            continue
            
        # Compter la fréquence des allèles
        counts = Counter(all_letters)
        alleles = [allele for allele, count in counts.most_common()]
        
        major_allele = alleles[0]
        minor_allele = alleles[1] if len(alleles) > 1 else major_allele # Fallback si monomorphique
        
        # Règle d'encodage standard (sans fichier de référence) : 
        # Homozygote Majeur = 0, Hétérozygote = 1, Homozygote Mineur = 2
        mapping = {
            f"{major_allele}{major_allele}": 0,
            f"{major_allele}{minor_allele}": 1,
            f"{minor_allele}{major_allele}": 1,
            f"{minor_allele}{minor_allele}": 2
        }
        
        # Appliquer le mapping, les valeurs non reconnues deviennent 0 (imputation de base)
        geo_numeric[feature] = raw_genotypes.map(mapping).fillna(0)

print(f"-> Match trouvé pour {matched_count} sur {len(expected_snps)} SNPs attendus.")

print("\n--- 5. INTÉGRATION ET NORMALISATION DES COVARIABLES ---")
# Aligner l'Age et le Gender
geo_numeric['Age'] = pheno_geo['Age'].reindex(geo_numeric.index)
geo_numeric['Gender'] = pheno_geo['Gender'].reindex(geo_numeric.index)

# Remplir les âges manquants (s'il y en a) avec la moyenne pour éviter un crash
geo_numeric['Age'] = geo_numeric['Age'].fillna(geo_numeric['Age'].mean())

# Appliquer le MinMaxScaler EXACT qui a été "fitté" sur les données d'entraînement
geo_numeric['Age'] = scaler.transform(geo_numeric[['Age']])

print("\n--- 6. ÉVALUATION FINALE DU MODÈLE XGBOOST ---")
# Récupérer la cible (y)
y_test_geo = pheno_geo.reindex(geo_numeric.index)['PHENOTYPE_NEW']

# Filtrer les échantillons valides (sans NAs dans la cible)
valid_indices = y_test_geo.dropna().index
X_test_geo_final = geo_numeric.loc[valid_indices]
y_test_geo_final = y_test_geo.loc[valid_indices]

# Prédiction
geo_preds = final_model.predict_proba(X_test_geo_final)[:, 1]
geo_auc = roc_auc_score(y_test_geo_final, geo_preds)

print("=====================================================================")
print(f" EXTERNAL VALIDATION AUC (XGBoost - GSE148812): {geo_auc:.3f}")
print("=====================================================================")

--- 1. EXTRACTION DES PHÉNOTYPES ET COVARIABLES ---
Chargement réussi : 1619 labels phénotypiques avec Âge et Genre.

--- 2. CHARGEMENT DU DICTIONNAIRE DE PROBES (GPL) ---
Dictionnaire chargé : 242924 sondes.

--- 3. CHARGEMENT ET TRANSPOSITION DE LA MATRICE GEO ---

--- 4. ENCODAGE DYNAMIQUE (MAJOR/MINOR) & ALIGNEMENT ---
-> Match trouvé pour 188 sur 188 SNPs attendus.

--- 5. INTÉGRATION ET NORMALISATION DES COVARIABLES ---

--- 6. ÉVALUATION FINALE DU MODÈLE XGBOOST ---


ValueError: feature_names mismatch: ['exm-rs1055388-131_B_R_1990478034', 'exm-rs11605924-131_B_R_1990478586', 'exm-rs12793173-131_T_R_1990488308', 'exm-rs1633017-131_B_R_1990479888', 'exm-rs1633036-131_B_R_1990479892', 'exm-rs16982520-131_T_F_1990489099', 'exm-rs1751492-131_B_F_1990480221', 'exm-rs2032672-131_B_R_1990480579', 'exm-rs2032673-131_B_F_2058870696', 'exm-rs2300478-131_T_R_1990490227', 'exm-rs2357013-131_B_F_1990481237', 'exm-rs2517713-131_B_R_1990481493', 'exm-rs2517723-131_T_F_1990490523', 'exm-rs2523393-131_T_R_1990490536', 'exm-rs2523987-131_T_R_1990490600', 'exm-rs2690775-131_B_R_1990481734', 'exm-rs2857105-131_B_R_1990482043', 'exm-rs2895811-131_T_R_1990491146', 'exm-rs3093978-131_B_F_1990482224', 'exm-rs3093992-131_B_F_1990482226', 'exm-rs3130055-131_B_F_1990482441', 'exm-rs3131628-131_B_R_1990482545', 'exm-rs3135031-132_T_R_1990486720', 'exm-rs3763365-131_T_R_1990491821', 'exm-rs4795519-131_B_R_1990483578', 'exm-rs6687758-131_T_F_1990493250', 'exm-rs7009183-131_B_R_1

In [23]:
print("\n--- 6. ÉVALUATION FINALE DU MODÈLE XGBOOST ---")
# Récupérer la cible (y)
y_test_geo = pheno_geo.reindex(geo_numeric.index)['PHENOTYPE_NEW']

# Filtrer les échantillons valides (sans NAs dans la cible)
valid_indices = y_test_geo.dropna().index
X_test_geo_final = geo_numeric.loc[valid_indices]
y_test_geo_final = y_test_geo.loc[valid_indices]

# LE CORRECTIF EST ICI : 
# 1. On force X_test_geo à avoir EXACTEMENT les mêmes colonnes, dans le même ordre que X_train_lasso
# 2. On convertit tout en float pour éviter que XGBoost ne panique sur des types "object"
X_test_geo_final = X_test_geo_final[X_train_lasso.columns].astype(float)

# Prédiction
geo_preds = final_model.predict_proba(X_test_geo_final)[:, 1]
geo_auc = roc_auc_score(y_test_geo_final, geo_preds)

print("=====================================================================")
print(f" EXTERNAL VALIDATION AUC (XGBoost - GSE148812): {geo_auc:.3f}")
print("=====================================================================")


--- 6. ÉVALUATION FINALE DU MODÈLE XGBOOST ---
 EXTERNAL VALIDATION AUC (XGBoost - GSE148812): 0.556
